In [1]:
from geneformer import EmbExtractor
from transformers import AutoModel, AutoTokenizer

/home/vlad/miniconda3/envs/bioml/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
model = AutoModel.from_pretrained("ctheodoris/Geneformer")
print(model)

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at ctheodoris/Geneformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(20275, 1152, padding_idx=0)
    (position_embeddings): Embedding(4096, 1152)
    (token_type_embeddings): Embedding(2, 1152)
    (LayerNorm): LayerNorm((1152,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-17): 18 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1152, out_features=1152, bias=True)
            (key): Linear(in_features=1152, out_features=1152, bias=True)
            (value): Linear(in_features=1152, out_features=1152, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1152, out_features=1152, bias=True)
            (LayerNorm): LayerNorm((1152,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1,

In [4]:
print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Параметров: 316.3M


In [ ]:
import scanpy as sc
import scipy.sparse as sp
import os

adata = sc.read_h5ad("data/raw/pbmc3k/pbmc3k_raw.h5ad")

print(adata.var.columns.tolist())  
print(adata.obs.columns.tolist())

In [ ]:
adata.var.rename(columns={'gene_ids': 'ensembl_id'}, inplace=True)

if sp.issparse(adata.X):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
else:
    adata.obs['n_counts'] = adata.X.sum(axis=1)

adata.write_h5ad('data/geneformer_input/pbmc3k/pbmc3k_geneformer.h5ad')

In [ ]:
from geneformer import TranscriptomeTokenizer


tk = TranscriptomeTokenizer()
tk.tokenize_data(
    data_directory = "data/geneformer_input/pbmc3k",
    output_directory = "tokenized",
    output_prefix = "pbmc3k",
    file_format = "h5ad"
)

In [ ]:
import scipy.io as sio
import pandas as pd
import anndata as ad
import numpy as np

raw_sca = "data/raw/pbmcsca"
counts = sio.mmread(f"{raw_sca}/pbmcsca_counts.mtx").tocsr()
genes = pd.read_csv(f"{raw_sca}/pbmcsca_genes.csv")["gene"].values
meta = pd.read_csv(f"{raw_sca}/pbmcsca_metadata.csv", index_col=0)

adata_sca = ad.AnnData(X=counts.T.tocsr(), obs=meta)
adata_sca.var_names = genes
adata_sca.var["gene_symbol"] = genes

print(adata_sca)
print(adata_sca.obs["Method"].value_counts())

In [ ]:
import pickle, os

name_id = pickle.load(open(
    "/home/vlad/miniconda3/envs/bioml/lib/python3.11/site-packages/geneformer/gene_name_id_dict_gc104M.pkl", "rb"))

adata_sca.var["ensembl_id"] = [name_id.get(g, None) for g in adata_sca.var["gene_symbol"]]

mapped = adata_sca.var["ensembl_id"].notna()
print(f"mapped {mapped.sum()} / {adata_sca.n_vars} genes")

adata_sca = adata_sca[:, mapped].copy()
adata_sca.obs["n_counts"] = np.asarray(adata_sca.X.sum(axis=1)).ravel()

os.makedirs("data/geneformer_input/pbmcsca", exist_ok=True)
adata_sca.write_h5ad("data/geneformer_input/pbmcsca/pbmcsca_geneformer.h5ad")
print(adata_sca)

In [ ]:
from geneformer import TranscriptomeTokenizer

tk = TranscriptomeTokenizer(
    custom_attr_name_dict={
        "Method": "Method",
        "Experiment": "Experiment",
        "CellType": "CellType",
    }
)
tk.tokenize_data(
    data_directory="data/geneformer_input/pbmcsca",
    output_directory="tokenized",
    output_prefix="pbmcsca",
    file_format="h5ad",
)